In [3]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import os
import re
from collections import Counter, defaultdict

# =========================================
# DATASET PATHS
# =========================================

BASE_DIR = "../"

RESIZED_DIR = os.path.join(BASE_DIR, "resized_images")

IMAGE_BASE_DIR = os.path.join(
    BASE_DIR,
    "detection_dataset/images"
)

LABEL_BASE_DIR = os.path.join(
    BASE_DIR,
    "detection_dataset/labels"
)

SPLITS = ["train", "val", "test"]

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

# =========================================
# REPORT STORAGE
# =========================================

report_lines = []

# =========================================
# HELPERS
# =========================================

pattern = re.compile(r"(.+)_(\d+)")


def log(text=""):

    print(text)
    report_lines.append(str(text))


def get_image_files(folder):

    return [
        f for f in os.listdir(folder)
        if f.lower().endswith(IMAGE_EXTENSIONS)
    ]


def get_label_files(folder):

    return [
        f for f in os.listdir(folder)
        if f.endswith(".txt")
    ]


def extract_numbers(files):

    result = defaultdict(list)

    for file in files:

        name = os.path.splitext(file)[0]

        match = pattern.match(name)

        if match:

            prefix = match.group(1)
            number = int(match.group(2))

            result[prefix].append(number)

    return result


def find_missing_numbers(numbers):

    if not numbers:
        return []

    return sorted(
        set(range(min(numbers), max(numbers) + 1))
        - set(numbers)
    )


# =========================================
# TOTAL RESIZED IMAGES
# =========================================

log("\n========== RESIZED IMAGE REPORT ==========\n")

total_resized = 0

for folder in os.listdir(RESIZED_DIR):

    folder_path = os.path.join(
        RESIZED_DIR,
        folder
    )

    if not os.path.isdir(folder_path):
        continue

    count = len(get_image_files(folder_path))

    total_resized += count

    log(f"{folder}: {count}")

log(f"\nTotal Resized Images: {total_resized}")

# =========================================
# GLOBAL COUNTERS
# =========================================

total_images = 0
total_labels = 0
total_annotations = 0

class_counter = Counter()

missing_labels = defaultdict(list)
missing_images = defaultdict(list)
empty_labels = defaultdict(list)
invalid_labels = defaultdict(list)

split_stats = {}

# =========================================
# SPLIT CHECK
# =========================================

for split in SPLITS:

    IMAGE_DIR = os.path.join(
        IMAGE_BASE_DIR,
        split
    )

    LABEL_DIR = os.path.join(
        LABEL_BASE_DIR,
        split
    )

    image_files = get_image_files(IMAGE_DIR)
    label_files = get_label_files(LABEL_DIR)

    image_names = {
        os.path.splitext(f)[0]
        for f in image_files
    }

    label_names = {
        os.path.splitext(f)[0]
        for f in label_files
    }

    # -------------------------------------
    # COUNT
    # -------------------------------------

    image_count = len(image_files)
    label_count = len(label_files)

    total_images += image_count
    total_labels += label_count

    # -------------------------------------
    # MISSING LABELS
    # -------------------------------------

    for img in image_names:

        if img not in label_names:

            missing_labels[split].append(img)

    # -------------------------------------
    # MISSING IMAGES
    # -------------------------------------

    for lbl in label_names:

        if lbl not in image_names:

            missing_images[split].append(lbl)

    # -------------------------------------
    # LABEL CHECK
    # -------------------------------------

    for label_file in label_files:

        label_path = os.path.join(
            LABEL_DIR,
            label_file
        )

        # Empty label
        if os.path.getsize(label_path) == 0:

            empty_labels[split].append(
                label_file
            )

            continue

        with open(label_path, "r") as f:

            lines = f.readlines()

        total_annotations += len(lines)

        for line in lines:

            parts = line.strip().split()

            # Invalid YOLO format
            if len(parts) != 5:

                invalid_labels[split].append(
                    label_file
                )

                continue

            class_id = parts[0]

            class_counter[class_id] += 1

    # -------------------------------------
    # NUMBER CHECK
    # -------------------------------------

    image_number_data = extract_numbers(
        image_files
    )

    missing_number_report = {}

    for cls, nums in image_number_data.items():

        missing_number_report[cls] = (
            find_missing_numbers(nums)
        )

    split_stats[split] = {
        "images": image_count,
        "labels": label_count,
        "missing_numbers": missing_number_report
    }

# =========================================
# FINAL REPORT
# =========================================

log("\n\n========== DATASET OVERVIEW ==========")

log(f"\nTotal Dataset Images : {total_images}")
log(f"Total Label Files    : {total_labels}")
log(f"Total Annotations    : {total_annotations}")

# =========================================
# SPLIT DISTRIBUTION
# =========================================

log("\n========== SPLIT DISTRIBUTION ==========")

for split in SPLITS:

    log(f"\n{split.upper()}")

    log(
        f"Images : "
        f"{split_stats[split]['images']}"
    )

    log(
        f"Labels : "
        f"{split_stats[split]['labels']}"
    )

# =========================================
# LABELING STATUS
# =========================================

log("\n========== LABELING STATUS ==========")

for split in SPLITS:

    count = len(missing_labels[split])

    log(f"\n{split.upper()}")

    log(f"Need Annotation : {count}")

    if count > 0:

        log("\nMissing Labels:")

        for item in sorted(
            missing_labels[split]
        ):
            log(f"  - {item}")

# =========================================
# NUMBER CHECK
# =========================================

log("\n========== MISSING NUMBERS ==========")

for split in SPLITS:

    log(f"\n{split.upper()}")

    report = split_stats[split][
        "missing_numbers"
    ]

    for cls, nums in report.items():

        log(f"\n{cls}")

        if nums:

            log(
                f"Missing Numbers: {nums}"
            )

        else:

            log(
                "No Missing Numbers"
            )

# =========================================
# CLASS DISTRIBUTION
# =========================================

log("\n========== CLASS DISTRIBUTION ==========")

log(
    f"Healthy (0)   : "
    f"{class_counter['0']}"
)

log(
    f"Unhealthy (1) : "
    f"{class_counter['1']}"
)

# =========================================
# DATASET HEALTH
# =========================================

log("\n========== DATASET HEALTH ==========")

for split in SPLITS:

    log(f"\n{split.upper()}")

    log(
        f"Missing Images : "
        f"{len(missing_images[split])}"
    )

    log(
        f"Empty Labels   : "
        f"{len(empty_labels[split])}"
    )

    log(
        f"Invalid Labels : "
        f"{len(invalid_labels[split])}"
    )

# =========================================
# ANNOTATION COMPLETION
# =========================================

annotated_images = (
    total_images
    - sum(
        len(missing_labels[s])
        for s in SPLITS
    )
)

completion = (
    annotated_images / total_images
) * 100

log("\n========== ANNOTATION PROGRESS ==========")

log(
    f"\nAnnotated Images : "
    f"{annotated_images}"
)

log(
    f"Remaining Images : "
    f"{total_images - annotated_images}"
)

log(
    f"Completion Rate  : "
    f"{completion:.2f}%"
)

# =========================================
# SAVE REPORT
# =========================================

report_path = "dataset_report.txt"

with open(report_path, "w") as f:

    f.write("\n".join(report_lines))

log("\n========== DATASET CHECK COMPLETED ==========")

log(f"\nReport Saved: {report_path}")


========== RESIZED IMAGE REPORT ==========

healthy: 138
healthy01: 305
unhealthy: 360
unhealthy01: 90

Total Resized Images: 893


========== DATASET OVERVIEW ==========

Total Dataset Images : 893
Total Label Files    : 896
Total Annotations    : 1769

========== SPLIT DISTRIBUTION ==========

TRAIN
Images : 622
Labels : 623

VAL
Images : 179
Labels : 180

TEST
Images : 92
Labels : 93

========== LABELING STATUS ==========

TRAIN
Need Annotation : 0

VAL
Need Annotation : 0

TEST
Need Annotation : 0

========== MISSING NUMBERS ==========

TRAIN

healthy
Missing Numbers: [97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138]

unhealthy
Missing Numbers: [252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288

5308


========== DATASET CHECK COMPLETED ==========

Report Saved: dataset_report.txt
